In [2]:
# print(f"\n=== Evaluating checkpoint: {"/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongClip_detail_aware_ver5_3_no_refine/train/longclip/lr=1e-05_wd=0.01_refine=0.0002_batch=64_wl=200_logs=4.6052_64xb/ckpt/ddp-epoch7-05-02--16_52_45.pt"} ===")
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
from types import SimpleNamespace
from torch.utils.data import DataLoader
# Import your modules
from train import CLIP_Clean_Train
from docci import DocciDataset
# from dci import DciDataset
import os
from types import SimpleNamespace
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
# from dci import JsonDCIDataset
import clip
from train import CLIP_Clean_Train

class Urban1kDataset(Dataset):
    """
    Dataset for Urban1k: pairs each image/ caption by filename (without extension).
    """
    def __init__(self, root_dir, max_items=None, device='cuda'):
        self.image_dir   = os.path.join(root_dir, 'image')
        self.caption_dir = os.path.join(root_dir, 'caption')
        # all caption files (strip “.txt”)
        self.ids = sorted([
            fname[:-4] for fname in os.listdir(self.caption_dir)
            if fname.endswith('.txt')
        ])
        if max_items is not None:
            self.ids = self.ids[:max_items]

        # load CLIP preprocess
        self.device     = torch.device(device)
        _, self.preprocess = clip.load('ViT-B/16', device=self.device)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        idx = self.ids[idx]
        # load caption
        with open(os.path.join(self.caption_dir, f'{idx}.txt'), 'r', encoding='utf8') as f:
            caption = f.read().strip().replace('\n', ' ')
        # load & preprocess image
        img_path = os.path.join(self.image_dir, f'{idx}.jpg')
        image    = Image.open(img_path).convert('RGB')
        image_t  = self.preprocess(image)
        return image_t, caption, "None", img_path, "None"

# Inline args for each run
args = SimpleNamespace(
    base_model='ViT-B/16',
    download_root=None,
    log_scale=4.6052,
    batch_size=64,
    epochs=1,
    lr=1e-5,
    refine_lr_image    = 2e-6,
    refine_lr_text = 2e-4,
    weight_decay=0.01,
    warmup_length=200,
    exp_name='eval_run',
    ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-21--16_36_34_-docci-spm-no-divide.pt"
)

# Instantiate and load model
trainer = CLIP_Clean_Train(args)
state = torch.load("/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-21--16_36_34_-docci-spm-no-divide.pt", map_location='cpu')
trainer.model.load_state_dict(state, strict=False)

from torch.utils.data import ConcatDataset, DataLoader

# data_root = '/home/ubuntu/shared/hieu.tq/Urban1k/Urban1k'
# test_ds        = Urban1kDataset(data_root, max_items=None, device='cuda')
test_ds       = DocciDataset(split='test',      max_items=None)
# qual_test_ds  = Share4VDataset(split='qual_test', max_items=None)
# test_ds  = JsonDCIDataset(
#     json_path="/home/ubuntu/hieu.tq/Git/GOAL/datasets/DCI_test.json",
#     max_items=None
# )

# # 1) Gộp 2 dataset thành 1
# merged_ds = ConcatDataset([test_ds, qual_test_ds])

# 2) Tạo loader cho dataset đã gộp
merged_loader = DataLoader(
    test_ds,
    batch_size   = args.batch_size,
    shuffle      = False,
    num_workers  = 32,
    pin_memory   = True,
)

print(f"Số mẫu trong test set: {len(merged_loader.dataset)}")
print(f"Số batch trong test loader: {len(merged_loader)}")

# Evaluate
metrics = trainer.test_epoch_ver4(merged_loader, epoch=0)
# mean_acc = sum(metrics) / len(metrics)

# Print results
print(f"Individual metrics: {metrics}")
# print(f"Mean accuracy: {mean_acc:.4%}")

Số mẫu trong test set: 5100
Số batch trong test loader: 80


Extracting features: 100%|██████████| 80/80 [02:05<00:00,  1.57s/it]


▶ Full Text→Image:     77.6667%
▶ Image→Full Text:     78.0784%
▶ 1th Sentence→Image:  51.0784%
▶ 2th Sentence→Image:  24.8627%
▶ 3th Sentence→Image:  19.0588%
▶ 4th Sentence→Image:  14.8431%
Individual metrics: [0.7766666666666666, 0.7807843137254902, 0.5107843137254902, 0.24862745098039216, 0.19058823529411764, 0.1484313725490196]


In [3]:
# print(f"\n=== Evaluating checkpoint: {"/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongClip_detail_aware_ver5_3_no_refine/train/longclip/lr=1e-05_wd=0.01_refine=0.0002_batch=64_wl=200_logs=4.6052_64xb/ckpt/ddp-epoch7-05-02--16_52_45.pt"} ===")
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
from types import SimpleNamespace
from torch.utils.data import DataLoader
# Import your modules
from train import CLIP_Clean_Train
from docci import DocciDataset
# from dci import DciDataset
import os
from types import SimpleNamespace
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
# from dci import JsonDCIDataset
import clip
from train import CLIP_Clean_Train

class Urban1kDataset(Dataset):
    """
    Dataset for Urban1k: pairs each image/ caption by filename (without extension).
    """
    def __init__(self, root_dir, max_items=None, device='cuda'):
        self.image_dir   = os.path.join(root_dir, 'image')
        self.caption_dir = os.path.join(root_dir, 'caption')
        # all caption files (strip “.txt”)
        self.ids = sorted([
            fname[:-4] for fname in os.listdir(self.caption_dir)
            if fname.endswith('.txt')
        ])
        if max_items is not None:
            self.ids = self.ids[:max_items]

        # load CLIP preprocess
        self.device     = torch.device(device)
        _, self.preprocess = clip.load('ViT-B/16', device=self.device)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        idx = self.ids[idx]
        # load caption
        with open(os.path.join(self.caption_dir, f'{idx}.txt'), 'r', encoding='utf8') as f:
            caption = f.read().strip().replace('\n', ' ')
        # load & preprocess image
        img_path = os.path.join(self.image_dir, f'{idx}.jpg')
        image    = Image.open(img_path).convert('RGB')
        image_t  = self.preprocess(image)
        return image_t, caption, "None", img_path, "None"

# Inline args for each run
args = SimpleNamespace(
    base_model='ViT-B/16',
    download_root=None,
    log_scale=4.6052,
    batch_size=64,
    epochs=1,
    lr=1e-5,
    refine_lr_image    = 2e-6,
    refine_lr_text = 2e-4,
    weight_decay=0.01,
    warmup_length=200,
    exp_name='eval_run',
    ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-21--16_36_34_-docci-spm-no-divide.pt"
)

# Instantiate and load model
trainer = CLIP_Clean_Train(args)
state = torch.load("/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-21--16_36_34_-docci-spm-no-divide.pt", map_location='cpu')
trainer.model.load_state_dict(state, strict=False)

from torch.utils.data import ConcatDataset, DataLoader

data_root = '/home/ubuntu/shared/hieu.tq/Urban1k/Urban1k'
test_ds        = Urban1kDataset(data_root, max_items=None, device='cuda')
# test_ds       = DocciDataset(split='test',      max_items=None)
# qual_test_ds  = Share4VDataset(split='qual_test', max_items=None)
# test_ds  = JsonDCIDataset(
#     json_path="/home/ubuntu/hieu.tq/Git/GOAL/datasets/DCI_test.json",
#     max_items=None
# )

# # 1) Gộp 2 dataset thành 1
# merged_ds = ConcatDataset([test_ds, qual_test_ds])

# 2) Tạo loader cho dataset đã gộp
merged_loader = DataLoader(
    test_ds,
    batch_size   = args.batch_size,
    shuffle      = False,
    num_workers  = 32,
    pin_memory   = True,
)

print(f"Số mẫu trong test set: {len(merged_loader.dataset)}")
print(f"Số batch trong test loader: {len(merged_loader)}")

# Evaluate
metrics = trainer.test_epoch_ver4(merged_loader, epoch=0)
# mean_acc = sum(metrics) / len(metrics)

# Print results
print(f"Individual metrics: {metrics}")
# print(f"Mean accuracy: {mean_acc:.4%}")

Số mẫu trong test set: 1000
Số batch trong test loader: 16


Extracting features: 100%|██████████| 16/16 [00:26<00:00,  1.66s/it]

▶ Full Text→Image:     73.5000%
▶ Image→Full Text:     80.8000%
▶ 1th Sentence→Image:  35.9000%
▶ 2th Sentence→Image:  28.4000%
▶ 3th Sentence→Image:  14.9000%
▶ 4th Sentence→Image:  9.4000%
Individual metrics: [0.735, 0.808, 0.359, 0.284, 0.149, 0.094]


In [9]:
# print(f"\n=== Evaluating checkpoint: {"/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongClip_detail_aware_ver5_3_no_refine/train/longclip/lr=1e-05_wd=0.01_refine=0.0002_batch=64_wl=200_logs=4.6052_64xb/ckpt/ddp-epoch7-05-02--16_52_45.pt"} ===")
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
from types import SimpleNamespace
from torch.utils.data import DataLoader
# Import your modules
from train import CLIP_Clean_Train
from docci import DocciDataset
# from dci import DciDataset
import os
from types import SimpleNamespace
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
# from dci import JsonDCIDataset
import clip
from train import CLIP_Clean_Train

class Urban1kDataset(Dataset):
    """
    Dataset for Urban1k: pairs each image/ caption by filename (without extension).
    """
    def __init__(self, root_dir, max_items=None, device='cuda'):
        self.image_dir   = os.path.join(root_dir, 'image')
        self.caption_dir = os.path.join(root_dir, 'caption')
        # all caption files (strip “.txt”)
        self.ids = sorted([
            fname[:-4] for fname in os.listdir(self.caption_dir)
            if fname.endswith('.txt')
        ])
        if max_items is not None:
            self.ids = self.ids[:max_items]

        # load CLIP preprocess
        self.device     = torch.device(device)
        _, self.preprocess = clip.load('ViT-B/16', device=self.device)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        idx = self.ids[idx]
        # load caption
        with open(os.path.join(self.caption_dir, f'{idx}.txt'), 'r', encoding='utf8') as f:
            caption = f.read().strip().replace('\n', ' ')
        # load & preprocess image
        img_path = os.path.join(self.image_dir, f'{idx}.jpg')
        image    = Image.open(img_path).convert('RGB')
        image_t  = self.preprocess(image)
        return image_t, caption, "None", img_path, "None"

# Inline args for each run
args = SimpleNamespace(
    base_model='ViT-B/16',
    download_root=None,
    log_scale=4.6052,
    batch_size=64,
    epochs=1,
    lr=1e-5,
    refine_lr_image    = 2e-6,
    refine_lr_text = 2e-4,
    weight_decay=0.01,
    warmup_length=200,
    exp_name='eval_run',
    ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-21--15_09_30_-dci-spm-no-divide-9.pt"
)

# Instantiate and load model
trainer = CLIP_Clean_Train(args)
state = torch.load("/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-21--15_09_30_-dci-spm-no-divide-9.pt", map_location='cpu')
trainer.model.load_state_dict(state, strict=False)

from torch.utils.data import ConcatDataset, DataLoader

data_root = '/home/ubuntu/shared/hieu.tq/Urban1k/Urban1k'
test_ds        = Urban1kDataset(data_root, max_items=None, device='cuda')
# test_ds       = DocciDataset(split='test',      max_items=None)
# qual_test_ds  = Share4VDataset(split='qual_test', max_items=None)
# test_ds  = JsonDCIDataset(
#     json_path="/home/ubuntu/hieu.tq/Git/GOAL/datasets/DCI_test.json",
#     max_items=None
# )

# # 1) Gộp 2 dataset thành 1
# merged_ds = ConcatDataset([test_ds, qual_test_ds])

# 2) Tạo loader cho dataset đã gộp
merged_loader = DataLoader(
    test_ds,
    batch_size   = args.batch_size,
    shuffle      = False,
    num_workers  = 32,
    pin_memory   = True,
)

print(f"Số mẫu trong test set: {len(merged_loader.dataset)}")
print(f"Số batch trong test loader: {len(merged_loader)}")

# Evaluate
metrics = trainer.test_epoch_ver4(merged_loader, epoch=0)
# mean_acc = sum(metrics) / len(metrics)

# Print results
print(f"Individual metrics: {metrics}")
# print(f"Mean accuracy: {mean_acc:.4%}")

Số mẫu trong test set: 1000
Số batch trong test loader: 16


Extracting features: 100%|██████████| 16/16 [00:18<00:00,  1.13s/it]

▶ Full Text→Image:     68.2000%
▶ Image→Full Text:     76.9000%
▶ 1th Sentence→Image:  32.5000%
▶ 2th Sentence→Image:  23.1000%
▶ 3th Sentence→Image:  14.3000%
▶ 4th Sentence→Image:  9.0000%
Individual metrics: [0.682, 0.769, 0.325, 0.231, 0.143, 0.09]


In [6]:
# print(f"\n=== Evaluating checkpoint: {"/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongClip_detail_aware_ver5_3_no_refine/train/longclip/lr=1e-05_wd=0.01_refine=0.0002_batch=64_wl=200_logs=4.6052_64xb/ckpt/ddp-epoch7-05-02--16_52_45.pt"} ===")
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
from types import SimpleNamespace
from torch.utils.data import DataLoader
# Import your modules
from train import CLIP_Clean_Train
from docci import DocciDataset
from dci import JsonDCIDataset
import os
from types import SimpleNamespace
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
# from dci import JsonDCIDataset
import clip
from train import CLIP_Clean_Train

class Urban1kDataset(Dataset):
    """
    Dataset for Urban1k: pairs each image/ caption by filename (without extension).
    """
    def __init__(self, root_dir, max_items=None, device='cuda'):
        self.image_dir   = os.path.join(root_dir, 'image')
        self.caption_dir = os.path.join(root_dir, 'caption')
        # all caption files (strip “.txt”)
        self.ids = sorted([
            fname[:-4] for fname in os.listdir(self.caption_dir)
            if fname.endswith('.txt')
        ])
        if max_items is not None:
            self.ids = self.ids[:max_items]

        # load CLIP preprocess
        self.device     = torch.device(device)
        _, self.preprocess = clip.load('ViT-B/16', device=self.device)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        idx = self.ids[idx]
        # load caption
        with open(os.path.join(self.caption_dir, f'{idx}.txt'), 'r', encoding='utf8') as f:
            caption = f.read().strip().replace('\n', ' ')
        # load & preprocess image
        img_path = os.path.join(self.image_dir, f'{idx}.jpg')
        image    = Image.open(img_path).convert('RGB')
        image_t  = self.preprocess(image)
        return image_t, caption, "None", img_path, "None"

# Inline args for each run
args = SimpleNamespace(
    base_model='ViT-B/16',
    download_root=None,
    log_scale=4.6052,
    batch_size=64,
    epochs=1,
    lr=1e-5,
    refine_lr_image    = 2e-6,
    refine_lr_text = 2e-4,
    weight_decay=0.01,
    warmup_length=200,
    exp_name='eval_run',
    ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-21--16_36_34_-docci-spm-no-divide.pt"
)

# Instantiate and load model
trainer = CLIP_Clean_Train(args)
state = torch.load("/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-21--16_36_34_-docci-spm-no-divide.pt", map_location='cpu')
trainer.model.load_state_dict(state, strict=False)

from torch.utils.data import ConcatDataset, DataLoader

# data_root = '/home/ubuntu/shared/hieu.tq/Urban1k/Urban1k'
# test_ds        = Urban1kDataset(data_root, max_items=None, device='cuda')
# test_ds       = DocciDataset(split='test',      max_items=None)
# qual_test_ds  = Share4VDataset(split='qual_test', max_items=None)
test_ds  = JsonDCIDataset(
    json_path="/home/ubuntu/hieu.tq/Git/GOAL/datasets/DCI_test.json",
    max_items=None
)

# # 1) Gộp 2 dataset thành 1
# merged_ds = ConcatDataset([test_ds, qual_test_ds])

# 2) Tạo loader cho dataset đã gộp
merged_loader = DataLoader(
    test_ds,
    batch_size   = args.batch_size,
    shuffle      = False,
    num_workers  = 32,
    pin_memory   = True,
)

print(f"Số mẫu trong test set: {len(merged_loader.dataset)}")
print(f"Số batch trong test loader: {len(merged_loader)}")

# Evaluate
metrics = trainer.test_epoch_ver4(merged_loader, epoch=0)
# mean_acc = sum(metrics) / len(metrics)

# Print results
print(f"Individual metrics: {metrics}")
# print(f"Mean accuracy: {mean_acc:.4%}")

Số mẫu trong test set: 1999
Số batch trong test loader: 32


Extracting features: 100%|██████████| 32/32 [00:54<00:00,  1.71s/it]

▶ Full Text→Image:     64.4322%
▶ Image→Full Text:     66.2331%
▶ 1th Sentence→Image:  35.8679%
▶ 2th Sentence→Image:  19.1596%
▶ 3th Sentence→Image:  16.0580%
▶ 4th Sentence→Image:  12.3062%
Individual metrics: [0.6443221610805403, 0.6623311655827914, 0.3586793396698349, 0.19159579789894948, 0.16058029014507252, 0.12306153076538269]


In [10]:
# print(f"\n=== Evaluating checkpoint: {"/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongClip_detail_aware_ver5_3_no_refine/train/longclip/lr=1e-05_wd=0.01_refine=0.0002_batch=64_wl=200_logs=4.6052_64xb/ckpt/ddp-epoch7-05-02--16_52_45.pt"} ===")
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
from types import SimpleNamespace
from torch.utils.data import DataLoader
# Import your modules
from train import CLIP_Clean_Train
from docci import DocciDataset
from dci import JsonDCIDataset
import os
from types import SimpleNamespace
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
# from dci import JsonDCIDataset
import clip
from train import CLIP_Clean_Train

class Urban1kDataset(Dataset):
    """
    Dataset for Urban1k: pairs each image/ caption by filename (without extension).
    """
    def __init__(self, root_dir, max_items=None, device='cuda'):
        self.image_dir   = os.path.join(root_dir, 'image')
        self.caption_dir = os.path.join(root_dir, 'caption')
        # all caption files (strip “.txt”)
        self.ids = sorted([
            fname[:-4] for fname in os.listdir(self.caption_dir)
            if fname.endswith('.txt')
        ])
        if max_items is not None:
            self.ids = self.ids[:max_items]

        # load CLIP preprocess
        self.device     = torch.device(device)
        _, self.preprocess = clip.load('ViT-B/16', device=self.device)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        idx = self.ids[idx]
        # load caption
        with open(os.path.join(self.caption_dir, f'{idx}.txt'), 'r', encoding='utf8') as f:
            caption = f.read().strip().replace('\n', ' ')
        # load & preprocess image
        img_path = os.path.join(self.image_dir, f'{idx}.jpg')
        image    = Image.open(img_path).convert('RGB')
        image_t  = self.preprocess(image)
        return image_t, caption, "None", img_path, "None"

# Inline args for each run
args = SimpleNamespace(
    base_model='ViT-B/16',
    download_root=None,
    log_scale=4.6052,
    batch_size=64,
    epochs=1,
    lr=1e-5,
    refine_lr_image    = 2e-6,
    refine_lr_text = 2e-4,
    weight_decay=0.01,
    warmup_length=200,
    exp_name='eval_run',
    ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-22--06_30_36_-dci-spm-no-divide-9.pt"
)

# Instantiate and load model
trainer = CLIP_Clean_Train(args)
state = torch.load("/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-22--06_30_36_-dci-spm-no-divide-9.pt", map_location='cpu')
trainer.model.load_state_dict(state, strict=False)

from torch.utils.data import ConcatDataset, DataLoader

# data_root = '/home/ubuntu/shared/hieu.tq/Urban1k/Urban1k'
# test_ds        = Urban1kDataset(data_root, max_items=None, device='cuda')
# test_ds       = DocciDataset(split='test',      max_items=None)
# qual_test_ds  = Share4VDataset(split='qual_test', max_items=None)
test_ds  = JsonDCIDataset(
    json_path="/home/ubuntu/hieu.tq/Git/GOAL/datasets/DCI_test.json",
    max_items=None
)

# # 1) Gộp 2 dataset thành 1
# merged_ds = ConcatDataset([test_ds, qual_test_ds])

# 2) Tạo loader cho dataset đã gộp
merged_loader = DataLoader(
    test_ds,
    batch_size   = args.batch_size,
    shuffle      = False,
    num_workers  = 32,
    pin_memory   = True,
)

print(f"Số mẫu trong test set: {len(merged_loader.dataset)}")
print(f"Số batch trong test loader: {len(merged_loader)}")

# Evaluate
metrics = trainer.test_epoch_ver4(merged_loader, epoch=0)
# mean_acc = sum(metrics) / len(metrics)

# Print results
print(f"Individual metrics: {metrics}")
# print(f"Mean accuracy: {mean_acc:.4%}")

Số mẫu trong test set: 1999
Số batch trong test loader: 32


Extracting features: 100%|██████████| 32/32 [00:36<00:00,  1.16s/it]

▶ Full Text→Image:     67.3337%
▶ Image→Full Text:     68.9845%
▶ 1th Sentence→Image:  36.5183%
▶ 2th Sentence→Image:  18.9595%
▶ 3th Sentence→Image:  16.1581%
▶ 4th Sentence→Image:  11.9560%
Individual metrics: [0.673336668334167, 0.6898449224612306, 0.36518259129564784, 0.18959479739869936, 0.1615807903951976, 0.11955977988994497]


In [11]:
# print(f"\n=== Evaluating checkpoint: {"/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongClip_detail_aware_ver5_3_no_refine/train/longclip/lr=1e-05_wd=0.01_refine=0.0002_batch=64_wl=200_logs=4.6052_64xb/ckpt/ddp-epoch7-05-02--16_52_45.pt"} ===")
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
from types import SimpleNamespace
from torch.utils.data import DataLoader
# Import your modules
from train import CLIP_Clean_Train
from docci import DocciDataset
from dci import JsonDCIDataset
import os
from types import SimpleNamespace
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
# from dci import JsonDCIDataset
import clip
from train import CLIP_Clean_Train

class Urban1kDataset(Dataset):
    """
    Dataset for Urban1k: pairs each image/ caption by filename (without extension).
    """
    def __init__(self, root_dir, max_items=None, device='cuda'):
        self.image_dir   = os.path.join(root_dir, 'image')
        self.caption_dir = os.path.join(root_dir, 'caption')
        # all caption files (strip “.txt”)
        self.ids = sorted([
            fname[:-4] for fname in os.listdir(self.caption_dir)
            if fname.endswith('.txt')
        ])
        if max_items is not None:
            self.ids = self.ids[:max_items]

        # load CLIP preprocess
        self.device     = torch.device(device)
        _, self.preprocess = clip.load('ViT-B/16', device=self.device)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        idx = self.ids[idx]
        # load caption
        with open(os.path.join(self.caption_dir, f'{idx}.txt'), 'r', encoding='utf8') as f:
            caption = f.read().strip().replace('\n', ' ')
        # load & preprocess image
        img_path = os.path.join(self.image_dir, f'{idx}.jpg')
        image    = Image.open(img_path).convert('RGB')
        image_t  = self.preprocess(image)
        return image_t, caption, "None", img_path, "None"

# Inline args for each run
args = SimpleNamespace(
    base_model='ViT-B/16',
    download_root=None,
    log_scale=4.6052,
    batch_size=64,
    epochs=1,
    lr=1e-5,
    refine_lr_image    = 2e-6,
    refine_lr_text = 2e-4,
    weight_decay=0.01,
    warmup_length=200,
    exp_name='eval_run',
    ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-22--11_14_58_-docci-spm-no-divide-9.pt"
)

# Instantiate and load model
trainer = CLIP_Clean_Train(args)
state = torch.load("/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-22--11_14_58_-docci-spm-no-divide-9.pt", map_location='cpu')
trainer.model.load_state_dict(state, strict=False)

from torch.utils.data import ConcatDataset, DataLoader

# data_root = '/home/ubuntu/shared/hieu.tq/Urban1k/Urban1k'
# test_ds        = Urban1kDataset(data_root, max_items=None, device='cuda')
# test_ds       = DocciDataset(split='test',      max_items=None)
# qual_test_ds  = Share4VDataset(split='qual_test', max_items=None)
test_ds  = JsonDCIDataset(
    json_path="/home/ubuntu/hieu.tq/Git/GOAL/datasets/DCI_test.json",
    max_items=None
)

# # 1) Gộp 2 dataset thành 1
# merged_ds = ConcatDataset([test_ds, qual_test_ds])

# 2) Tạo loader cho dataset đã gộp
merged_loader = DataLoader(
    test_ds,
    batch_size   = args.batch_size,
    shuffle      = False,
    num_workers  = 32,
    pin_memory   = True,
)

print(f"Số mẫu trong test set: {len(merged_loader.dataset)}")
print(f"Số batch trong test loader: {len(merged_loader)}")

# Evaluate
metrics = trainer.test_epoch_ver4(merged_loader, epoch=0)
# mean_acc = sum(metrics) / len(metrics)

# Print results
print(f"Individual metrics: {metrics}")
# print(f"Mean accuracy: {mean_acc:.4%}")

Số mẫu trong test set: 1999
Số batch trong test loader: 32


Extracting features: 100%|██████████| 32/32 [00:37<00:00,  1.17s/it]

▶ Full Text→Image:     64.2321%
▶ Image→Full Text:     66.0330%
▶ 1th Sentence→Image:  35.9680%
▶ 2th Sentence→Image:  19.2096%
▶ 3th Sentence→Image:  15.9580%
▶ 4th Sentence→Image:  12.6063%
Individual metrics: [0.6423211605802901, 0.6603301650825413, 0.35967983991996, 0.192096048024012, 0.15957978989494748, 0.12606303151575787]


In [13]:
# print(f"\n=== Evaluating checkpoint: {"/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongClip_detail_aware_ver5_3_no_refine/train/longclip/lr=1e-05_wd=0.01_refine=0.0002_batch=64_wl=200_logs=4.6052_64xb/ckpt/ddp-epoch7-05-02--16_52_45.pt"} ===")
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
from types import SimpleNamespace
from torch.utils.data import DataLoader
# Import your modules
from train import CLIP_Clean_Train
from docci import DocciDataset
from dci import JsonDCIDataset
import os
from types import SimpleNamespace
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
# from dci import JsonDCIDataset
import clip
from train import CLIP_Clean_Train

class Urban1kDataset(Dataset):
    """
    Dataset for Urban1k: pairs each image/ caption by filename (without extension).
    """
    def __init__(self, root_dir, max_items=None, device='cuda'):
        self.image_dir   = os.path.join(root_dir, 'image')
        self.caption_dir = os.path.join(root_dir, 'caption')
        # all caption files (strip “.txt”)
        self.ids = sorted([
            fname[:-4] for fname in os.listdir(self.caption_dir)
            if fname.endswith('.txt')
        ])
        if max_items is not None:
            self.ids = self.ids[:max_items]

        # load CLIP preprocess
        self.device     = torch.device(device)
        _, self.preprocess = clip.load('ViT-B/16', device=self.device)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        idx = self.ids[idx]
        # load caption
        with open(os.path.join(self.caption_dir, f'{idx}.txt'), 'r', encoding='utf8') as f:
            caption = f.read().strip().replace('\n', ' ')
        # load & preprocess image
        img_path = os.path.join(self.image_dir, f'{idx}.jpg')
        image    = Image.open(img_path).convert('RGB')
        image_t  = self.preprocess(image)
        return image_t, caption, "None", img_path, "None"

# Inline args for each run
args = SimpleNamespace(
    base_model='ViT-B/16',
    download_root=None,
    log_scale=4.6052,
    batch_size=64,
    epochs=1,
    lr=1e-5,
    refine_lr_image    = 2e-6,
    refine_lr_text = 2e-4,
    weight_decay=0.01,
    warmup_length=200,
    exp_name='eval_run',
    ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-22--11_14_58_-docci-spm-no-divide-9.pt"
)

# Instantiate and load model
trainer = CLIP_Clean_Train(args)
state = torch.load("/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-22--11_14_58_-docci-spm-no-divide-9.pt", map_location='cpu')
trainer.model.load_state_dict(state, strict=False)

from torch.utils.data import ConcatDataset, DataLoader

# data_root = '/home/ubuntu/shared/hieu.tq/Urban1k/Urban1k'
test_ds        = Urban1kDataset(data_root, max_items=None, device='cuda')
# test_ds       = DocciDataset(split='test',      max_items=None)
# qual_test_ds  = Share4VDataset(split='qual_test', max_items=None)
# test_ds  = JsonDCIDataset(
#     json_path="/home/ubuntu/hieu.tq/Git/GOAL/datasets/DCI_test.json",
#     max_items=None
# )

# # 1) Gộp 2 dataset thành 1
# merged_ds = ConcatDataset([test_ds, qual_test_ds])

# 2) Tạo loader cho dataset đã gộp
merged_loader = DataLoader(
    test_ds,
    batch_size   = args.batch_size,
    shuffle      = False,
    num_workers  = 32,
    pin_memory   = True,
)

print(f"Số mẫu trong test set: {len(merged_loader.dataset)}")
print(f"Số batch trong test loader: {len(merged_loader)}")

# Evaluate
metrics = trainer.test_epoch_ver4(merged_loader, epoch=0)
# mean_acc = sum(metrics) / len(metrics)

# Print results
print(f"Individual metrics: {metrics}")
# print(f"Mean accuracy: {mean_acc:.4%}")

Số mẫu trong test set: 1000
Số batch trong test loader: 16


Extracting features: 100%|██████████| 16/16 [00:18<00:00,  1.17s/it]

▶ Full Text→Image:     73.8000%
▶ Image→Full Text:     80.6000%
▶ 1th Sentence→Image:  35.8000%
▶ 2th Sentence→Image:  28.6000%
▶ 3th Sentence→Image:  15.1000%
▶ 4th Sentence→Image:  9.5000%
Individual metrics: [0.738, 0.806, 0.358, 0.286, 0.151, 0.095]


In [12]:
# print(f"\n=== Evaluating checkpoint: {"/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongClip_detail_aware_ver5_3_no_refine/train/longclip/lr=1e-05_wd=0.01_refine=0.0002_batch=64_wl=200_logs=4.6052_64xb/ckpt/ddp-epoch7-05-02--16_52_45.pt"} ===")
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
from types import SimpleNamespace
from torch.utils.data import DataLoader
# Import your modules
from train import CLIP_Clean_Train
from docci import DocciDataset
from dci import JsonDCIDataset
import os
from types import SimpleNamespace
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
# from dci import JsonDCIDataset
import clip
from train import CLIP_Clean_Train

class Urban1kDataset(Dataset):
    """
    Dataset for Urban1k: pairs each image/ caption by filename (without extension).
    """
    def __init__(self, root_dir, max_items=None, device='cuda'):
        self.image_dir   = os.path.join(root_dir, 'image')
        self.caption_dir = os.path.join(root_dir, 'caption')
        # all caption files (strip “.txt”)
        self.ids = sorted([
            fname[:-4] for fname in os.listdir(self.caption_dir)
            if fname.endswith('.txt')
        ])
        if max_items is not None:
            self.ids = self.ids[:max_items]

        # load CLIP preprocess
        self.device     = torch.device(device)
        _, self.preprocess = clip.load('ViT-B/16', device=self.device)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        idx = self.ids[idx]
        # load caption
        with open(os.path.join(self.caption_dir, f'{idx}.txt'), 'r', encoding='utf8') as f:
            caption = f.read().strip().replace('\n', ' ')
        # load & preprocess image
        img_path = os.path.join(self.image_dir, f'{idx}.jpg')
        image    = Image.open(img_path).convert('RGB')
        image_t  = self.preprocess(image)
        return image_t, caption, "None", img_path, "None"

# Inline args for each run
args = SimpleNamespace(
    base_model='ViT-B/16',
    download_root=None,
    log_scale=4.6052,
    batch_size=64,
    epochs=1,
    lr=1e-5,
    refine_lr_image    = 2e-6,
    refine_lr_text = 2e-4,
    weight_decay=0.01,
    warmup_length=200,
    exp_name='eval_run',
    ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-22--06_30_36_-dci-spm-no-divide-9.pt"
)

# Instantiate and load model
trainer = CLIP_Clean_Train(args)
state = torch.load("/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-22--06_30_36_-dci-spm-no-divide-9.pt", map_location='cpu')
trainer.model.load_state_dict(state, strict=False)

from torch.utils.data import ConcatDataset, DataLoader

# data_root = '/home/ubuntu/shared/hieu.tq/Urban1k/Urban1k'
# test_ds        = Urban1kDataset(data_root, max_items=None, device='cuda')
test_ds       = DocciDataset(split='test',      max_items=None)
# qual_test_ds  = Share4VDataset(split='qual_test', max_items=None)
# test_ds  = JsonDCIDataset(
#     json_path="/home/ubuntu/hieu.tq/Git/GOAL/datasets/DCI_test.json",
#     max_items=None
# )

# # 1) Gộp 2 dataset thành 1
# merged_ds = ConcatDataset([test_ds, qual_test_ds])

# 2) Tạo loader cho dataset đã gộp
merged_loader = DataLoader(
    test_ds,
    batch_size   = args.batch_size,
    shuffle      = False,
    num_workers  = 32,
    pin_memory   = True,
)

print(f"Số mẫu trong test set: {len(merged_loader.dataset)}")
print(f"Số batch trong test loader: {len(merged_loader)}")

# Evaluate
metrics = trainer.test_epoch_ver4(merged_loader, epoch=0)
# mean_acc = sum(metrics) / len(metrics)

# Print results
print(f"Individual metrics: {metrics}")
# print(f"Mean accuracy: {mean_acc:.4%}")

Số mẫu trong test set: 5100
Số batch trong test loader: 80


Extracting features: 100%|██████████| 80/80 [01:23<00:00,  1.04s/it]

▶ Full Text→Image:     67.2745%
▶ Image→Full Text:     68.3922%
▶ 1th Sentence→Image:  42.4314%
▶ 2th Sentence→Image:  21.8431%
▶ 3th Sentence→Image:  16.6275%
▶ 4th Sentence→Image:  12.8431%
Individual metrics: [0.6727450980392157, 0.683921568627451, 0.4243137254901961, 0.21843137254901962, 0.16627450980392156, 0.1284313725490196]


In [3]:
# print(f"\n=== Evaluating checkpoint: {"/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongClip_detail_aware_ver5_3_no_refine/train/longclip/lr=1e-05_wd=0.01_refine=0.0002_batch=64_wl=200_logs=4.6052_64xb/ckpt/ddp-epoch7-05-02--16_52_45.pt"} ===")
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
from types import SimpleNamespace
from torch.utils.data import DataLoader
# Import your modules
from train import CLIP_Clean_Train
from docci import DocciDataset
from dci import JsonDCIDataset
import os
from types import SimpleNamespace
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
# from dci import JsonDCIDataset
import clip
from train import CLIP_Clean_Train

class Urban1kDataset(Dataset):
    """
    Dataset for Urban1k: pairs each image/ caption by filename (without extension).
    """
    def __init__(self, root_dir, max_items=None, device='cuda'):
        self.image_dir   = os.path.join(root_dir, 'image')
        self.caption_dir = os.path.join(root_dir, 'caption')
        # all caption files (strip “.txt”)
        self.ids = sorted([
            fname[:-4] for fname in os.listdir(self.caption_dir)
            if fname.endswith('.txt')
        ])
        if max_items is not None:
            self.ids = self.ids[:max_items]

        # load CLIP preprocess
        self.device     = torch.device(device)
        _, self.preprocess = clip.load('ViT-B/16', device=self.device)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        idx = self.ids[idx]
        # load caption
        with open(os.path.join(self.caption_dir, f'{idx}.txt'), 'r', encoding='utf8') as f:
            caption = f.read().strip().replace('\n', ' ')
        # load & preprocess image
        img_path = os.path.join(self.image_dir, f'{idx}.jpg')
        image    = Image.open(img_path).convert('RGB')
        image_t  = self.preprocess(image)
        return image_t, caption, "None", img_path, "None"

# Inline args for each run
args = SimpleNamespace(
    base_model='ViT-B/16',
    download_root=None,
    log_scale=4.6052,
    batch_size=64,
    epochs=1,
    lr=1e-5,
    refine_lr_image    = 2e-6,
    refine_lr_text = 2e-4,
    weight_decay=0.01,
    warmup_length=200,
    exp_name='eval_run',
    ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-20--18_41_54_-docci-no.pt"
)

# Instantiate and load model
trainer = CLIP_Clean_Train(args)
state = torch.load("/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-20--18_41_54_-docci-no.pt", map_location='cpu')
trainer.model.load_state_dict(state, strict=False)

from torch.utils.data import ConcatDataset, DataLoader

data_root = '/home/ubuntu/shared/hieu.tq/Urban1k/Urban1k'
test_ds        = Urban1kDataset(data_root, max_items=None, device='cuda')
# test_ds       = DocciDataset(split='test',      max_items=None)
# qual_test_ds  = Share4VDataset(split='qual_test', max_items=None)
# test_ds  = JsonDCIDataset(
#     json_path="/home/ubuntu/hieu.tq/Git/GOAL/datasets/DCI_test.json",
#     max_items=None
# )

# # 1) Gộp 2 dataset thành 1
# merged_ds = ConcatDataset([test_ds, qual_test_ds])

# 2) Tạo loader cho dataset đã gộp
merged_loader = DataLoader(
    test_ds,
    batch_size   = args.batch_size,
    shuffle      = False,
    num_workers  = 32,
    pin_memory   = True,
)

print(f"Số mẫu trong test set: {len(merged_loader.dataset)}")
print(f"Số batch trong test loader: {len(merged_loader)}")

# Evaluate
metrics = trainer.test_epoch_ver4(merged_loader, epoch=0)
# mean_acc = sum(metrics) / len(metrics)

# Print results
print(f"Individual metrics: {metrics}")
# print(f"Mean accuracy: {mean_acc:.4%}")

Số mẫu trong test set: 1000
Số batch trong test loader: 16


Extracting features: 100%|██████████| 16/16 [00:16<00:00,  1.06s/it]

▶ Full Text→Image:     70.0000%
▶ Image→Full Text:     75.7000%
▶ 1th Sentence→Image:  32.6000%
▶ 2th Sentence→Image:  26.4000%
▶ 3th Sentence→Image:  14.5000%
▶ 4th Sentence→Image:  9.6000%
Individual metrics: [0.7, 0.757, 0.326, 0.264, 0.145, 0.096]


In [2]:
# print(f"\n=== Evaluating checkpoint: {"/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongClip_detail_aware_ver5_3_no_refine/train/longclip/lr=1e-05_wd=0.01_refine=0.0002_batch=64_wl=200_logs=4.6052_64xb/ckpt/ddp-epoch7-05-02--16_52_45.pt"} ===")
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
from types import SimpleNamespace
from torch.utils.data import DataLoader
# Import your modules
from train import CLIP_Clean_Train
from docci import DocciDataset
from dci import JsonDCIDataset
import os
from types import SimpleNamespace
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
# from dci import JsonDCIDataset
import clip
from train import CLIP_Clean_Train

class Urban1kDataset(Dataset):
    """
    Dataset for Urban1k: pairs each image/ caption by filename (without extension).
    """
    def __init__(self, root_dir, max_items=None, device='cuda'):
        self.image_dir   = os.path.join(root_dir, 'image')
        self.caption_dir = os.path.join(root_dir, 'caption')
        # all caption files (strip “.txt”)
        self.ids = sorted([
            fname[:-4] for fname in os.listdir(self.caption_dir)
            if fname.endswith('.txt')
        ])
        if max_items is not None:
            self.ids = self.ids[:max_items]

        # load CLIP preprocess
        self.device     = torch.device(device)
        _, self.preprocess = clip.load('ViT-B/16', device=self.device)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        idx = self.ids[idx]
        # load caption
        with open(os.path.join(self.caption_dir, f'{idx}.txt'), 'r', encoding='utf8') as f:
            caption = f.read().strip().replace('\n', ' ')
        # load & preprocess image
        img_path = os.path.join(self.image_dir, f'{idx}.jpg')
        image    = Image.open(img_path).convert('RGB')
        image_t  = self.preprocess(image)
        return image_t, caption, "None", img_path, "None"

# Inline args for each run
args = SimpleNamespace(
    base_model='ViT-B/16',
    download_root=None,
    log_scale=4.6052,
    batch_size=64,
    epochs=1,
    lr=1e-5,
    refine_lr_image    = 2e-6,
    refine_lr_text = 2e-4,
    weight_decay=0.01,
    warmup_length=200,
    exp_name='eval_run',
    ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-23--11_35_11_-dci-no-5.pt"
)

# Instantiate and load model
trainer = CLIP_Clean_Train(args)
state = torch.load("/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-23--11_35_11_-dci-no-5.pt", map_location='cpu')
trainer.model.load_state_dict(state, strict=False)

from torch.utils.data import ConcatDataset, DataLoader

data_root = '/home/ubuntu/shared/hieu.tq/Urban1k/Urban1k'
test_ds        = Urban1kDataset(data_root, max_items=None, device='cuda')
# test_ds       = DocciDataset(split='test',      max_items=None)
# qual_test_ds  = Share4VDataset(split='qual_test', max_items=None)
# test_ds  = JsonDCIDataset(
#     json_path="/home/ubuntu/hieu.tq/Git/GOAL/datasets/DCI_test.json",
#     max_items=None
# )

# # 1) Gộp 2 dataset thành 1
# merged_ds = ConcatDataset([test_ds, qual_test_ds])

# 2) Tạo loader cho dataset đã gộp
merged_loader = DataLoader(
    test_ds,
    batch_size   = args.batch_size,
    shuffle      = False,
    num_workers  = 32,
    pin_memory   = True,
)

print(f"Số mẫu trong test set: {len(merged_loader.dataset)}")
print(f"Số batch trong test loader: {len(merged_loader)}")

# Evaluate
metrics = trainer.test_epoch_ver4(merged_loader, epoch=0)
# mean_acc = sum(metrics) / len(metrics)

# Print results
print(f"Individual metrics: {metrics}")
# print(f"Mean accuracy: {mean_acc:.4%}")

Số mẫu trong test set: 1000
Số batch trong test loader: 16


Extracting features: 100%|██████████| 16/16 [00:18<00:00,  1.13s/it]

▶ Full Text→Image:     64.0000%
▶ Image→Full Text:     73.8000%
▶ 1th Sentence→Image:  30.7000%
▶ 2th Sentence→Image:  22.2000%
▶ 3th Sentence→Image:  13.0000%
▶ 4th Sentence→Image:  10.3000%
Individual metrics: [0.64, 0.738, 0.307, 0.222, 0.13, 0.103]


In [ ]:
# print(f"\n=== Evaluating checkpoint: {"/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongClip_detail_aware_ver5_3_no_refine/train/longclip/lr=1e-05_wd=0.01_refine=0.0002_batch=64_wl=200_logs=4.6052_64xb/ckpt/ddp-epoch7-05-02--16_52_45.pt"} ===")
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
from types import SimpleNamespace
from torch.utils.data import DataLoader
# Import your modules
from train import CLIP_Clean_Train
from docci import DocciDataset
from dci import JsonDCIDataset
import os
from types import SimpleNamespace
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
# from dci import JsonDCIDataset
import clip
from train import CLIP_Clean_Train

class Urban1kDataset(Dataset):
    """
    Dataset for Urban1k: pairs each image/ caption by filename (without extension).
    """
    def __init__(self, root_dir, max_items=None, device='cuda'):
        self.image_dir   = os.path.join(root_dir, 'image')
        self.caption_dir = os.path.join(root_dir, 'caption')
        # all caption files (strip “.txt”)
        self.ids = sorted([
            fname[:-4] for fname in os.listdir(self.caption_dir)
            if fname.endswith('.txt')
        ])
        if max_items is not None:
            self.ids = self.ids[:max_items]

        # load CLIP preprocess
        self.device     = torch.device(device)
        _, self.preprocess = clip.load('ViT-B/16', device=self.device)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        idx = self.ids[idx]
        # load caption
        with open(os.path.join(self.caption_dir, f'{idx}.txt'), 'r', encoding='utf8') as f:
            caption = f.read().strip().replace('\n', ' ')
        # load & preprocess image
        img_path = os.path.join(self.image_dir, f'{idx}.jpg')
        image    = Image.open(img_path).convert('RGB')
        image_t  = self.preprocess(image)
        return image_t, caption, "None", img_path, "None"

# Inline args for each run
args = SimpleNamespace(
    base_model='ViT-B/16',
    download_root=None,
    log_scale=4.6052,
    batch_size=64,
    epochs=1,
    lr=1e-5,
    refine_lr_image    = 2e-6,
    refine_lr_text = 2e-4,
    weight_decay=0.01,
    warmup_length=200,
    exp_name='eval_run',
    ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-22--06_30_36_-dci-spm-no-divide-9.pt"
)

# Instantiate and load model
trainer = CLIP_Clean_Train(args)
state = torch.load("/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-22--06_30_36_-dci-spm-no-divide-9.pt", map_location='cpu')
trainer.model.load_state_dict(state, strict=False)

from torch.utils.data import ConcatDataset, DataLoader

# data_root = '/home/ubuntu/shared/hieu.tq/Urban1k/Urban1k'
test_ds        = Urban1kDataset(data_root, max_items=None, device='cuda')
# test_ds       = DocciDataset(split='test',      max_items=None)
# qual_test_ds  = Share4VDataset(split='qual_test', max_items=None)
# test_ds  = JsonDCIDataset(
#     json_path="/home/ubuntu/hieu.tq/Git/GOAL/datasets/DCI_test.json",
#     max_items=None
# )

# # 1) Gộp 2 dataset thành 1
# merged_ds = ConcatDataset([test_ds, qual_test_ds])

# 2) Tạo loader cho dataset đã gộp
merged_loader = DataLoader(
    test_ds,
    batch_size   = args.batch_size,
    shuffle      = False,
    num_workers  = 32,
    pin_memory   = True,
)

print(f"Số mẫu trong test set: {len(merged_loader.dataset)}")
print(f"Số batch trong test loader: {len(merged_loader)}")

# Evaluate
metrics = trainer.test_epoch_ver4(merged_loader, epoch=0)
# mean_acc = sum(metrics) / len(metrics)

# Print results
print(f"Individual metrics: {metrics}")
# print(f"Mean accuracy: {mean_acc:.4%}")

Số mẫu trong test set: 1000
Số batch trong test loader: 16


Extracting features: 100%|██████████| 16/16 [00:19<00:00,  1.19s/it]

▶ Full Text→Image:     67.2000%
▶ Image→Full Text:     76.4000%
▶ 1th Sentence→Image:  32.3000%
▶ 2th Sentence→Image:  22.8000%
▶ 3th Sentence→Image:  14.3000%
▶ 4th Sentence→Image:  8.7000%
Individual metrics: [0.672, 0.764, 0.323, 0.228, 0.143, 0.087]


: 